In [1]:
import json
import duckdb
import requests
import pandas as pd

In [2]:
db_connection = duckdb.connect('loadsmart/dev.duckdb')

In [3]:
manifest = 'loadsmart/target/manifest.json'
with open(manifest, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

In [4]:
table_content = []
for item_id, item in manifest.get('nodes', {}).items():
    if item.get('resource_type') == 'model':
        table = item.get('name')
        table_ds = item.get('description', '')
        columns = item.get('columns', {})
        columns_ds = [f" - {col_name}: {col_info.get('description', '')}" for col_name, col_info in columns.items()]
        table_content.append(f"Table: {table}\nDescription: {table_ds}\nColumns:\n" + "\n".join(columns_ds))

In [5]:
ai_guide = "\n\n".join(table_content)

In [6]:
def ask_ai_db_question(question):
    ai_prompt = f"""
        You are an expert DuckDB SQL AI.

        SCHEMA METADATA (Generated from dbt):
        {ai_guide}

        CRITICAL BEHAVIORAL RULES:
        You must read and strictly obey all 'description' fields in the schema metadata above. They contain mandatory rules for handling historical dates and filtering metrics.
        Write a valid DuckDB SQL query to answer the question.
        Return only the executable SQL query in a markdown code block (```sql ... ```).

        Question: {question}
    """

    api_response = requests.post(
        "http://localhost:11434/api/generate",
        json = {
            'model': 'qwen2.5-coder:14b',
            'prompt': ai_prompt,
            'stream': False,
            'options': {'temperature': 0.0}
        }
    )

    llm_response = api_response.json()
    raw_sql = llm_response.get('response', '')
    cleaned_sql = raw_sql.replace('```sql', '').replace('```', '').strip()

    print(f"--- SQL ---\n{cleaned_sql}\n ---")

    # old code - no retry, crashes question 8 if the SQL is broken:
    # result_table = db_connection.execute(cleaned_sql).fetchdf()
    # return cleaned_sql, result_table

    # new code - if the SQL fails, send the error back to the model and try once more
    try:
        result_table = db_connection.execute(cleaned_sql).fetchdf()
    except Exception as e:
        fix_prompt = ai_prompt + f"\n\nThat query failed with this error:\n{e}\n\nFix it and return only the corrected SQL."
        api_response = requests.post(
            "http://localhost:11434/api/generate",
            json = {
                'model': 'qwen2.5-coder:14b',
                'prompt': fix_prompt,
                'stream': False,
                'options': {'temperature': 0.0}
            }
        )
        cleaned_sql = api_response.json().get('response', '').replace('```sql', '').replace('```', '').strip()
        print(f"--- retry SQL ---\n{cleaned_sql}\n ---")
        result_table = db_connection.execute(cleaned_sql).fetchdf()

    return cleaned_sql, result_table

In [7]:
original_question = ask_ai_db_question
results = []

def ask_ai_db_question(question):
    sql_query, result_df = original_question(question)
    results.append({
        "question": question,
        "generated_sql": sql_query,
        "answer": result_df.to_dict(orient="records"),
    })
    return sql_query, result_df

In [8]:
questions = [
    "How many loads were delivered in the last full month available in the data?",
    "Which shipper had the highest total book price?",
    "What is the average book price per load by pickup state?",
    "What are the top 5 lanes by number of delivered loads?",
    "Which carrier moved the most loads into Texas?",
    "How does the average book price compare between intrastate and interstate loads?",
    "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?",
    "Among lanes with at least 10 delivered loads, which had the highest average book price?",
]

for question in questions:
    print(f"QUESTION: {question}")
    try:
        sql_query, result_df = ask_ai_db_question(question)
        display(result_df)
    except Exception as e:
        print(f"FAILED: {e}")
    print()

QUESTION: How many loads were delivered in the last full month available in the data?


--- SQL ---
SELECT COUNT(*) AS delivered_loads
FROM fact_loadsmart
WHERE delivery_date >= DATE '2024-12-01' AND delivery_date < DATE '2025-01-01'
  AND load_was_cancelled = FALSE;
 ---


,delivered_loads
0,442



QUESTION: Which shipper had the highest total book price?


--- SQL ---
SELECT ds.shipper_name
FROM dim_shippers ds
JOIN fact_loadsmart fl ON ds.shipper_key = fl.shipper_key
WHERE fl.load_was_cancelled = FALSE
GROUP BY ds.shipper_name
ORDER BY SUM(fl.book_price) DESC
LIMIT 1;
 ---


...

--- SQL ---
WITH delivered_loads AS (
    SELECT
        f.shipper_key,
        DATE_TRUNC('month', f.delivery_date) AS month,
        COUNT(*) AS load_count
    FROM
        fact_loadsmart f
    WHERE
        f.load_was_cancelled = FALSE
    GROUP BY
        f.shipper_key,
        DATE_TRUNC('month', f.delivery_date)
),
shipper_load_counts AS (
    SELECT
        shipper_key,
        SUM(load_count) AS total_loads
    FROM
        delivered_loads
    GROUP BY
        shipper_key
),
top_shipper AS (
    SELECT
        shipper_key
    FROM
        shipper_load_counts
    ORDER BY
        total_loads DESC
    LIMIT 1
)
SELECT
    dl.month,
    dl.load_count
FROM
    delivered_loads dl
JOIN
    top_shipper ts ON dl.shipper_key = ts.shipper_key
ORDER BY
    dl.month;
 ---


,month,load_count
0,2024-01-01,127
1,2024-02-01,108
2,2024-03-01,178
3,2024-04-01,191
4,2024-05-01,163
5,2024-06-01,165
6,2024-07-01,137
7,2024-08-01,126
8,2024-09-01,137
9,2024-10-01,153



QUESTION: Among lanes with at least 10 delivered loads, which had the highest average book price?


--- SQL ---
WITH lane_book_prices AS (
    SELECT
        l.lane_key,
        AVG(f.book_price) AS avg_book_price,
        COUNT(f.loadsmart_id) AS delivered_load_count
    FROM
        fact_loadsmart f
    JOIN
        dim_lanes l ON f.lane_key = l.lane_key
    WHERE
        f.load_was_cancelled = FALSE
    GROUP BY
        l.lane_key
    HAVING
        COUNT(f.loadsmart_id) >= 10
)
SELECT
    l.lane,
    l.pickup_city,
    l.pickup_state,
    l.delivery_city,
    l.delivery_state,
    lbp.avg_book_price,
    lbp.delivered_load_count
FROM
    lane_book_prices lbp
JOIN
    dim_lanes l ON lbp.lane_key = l.lane_key
ORDER BY
    lbp.avg_book_price DESC
LIMIT 1;
 ---


,lane,pickup_city,pickup_state,delivery_city,delivery_state,avg_book_price,delivered_load_count
0,"Stockton,CA -> Parrish,FL",Stockton,CA,Parrish,FL,6800.0,11


In [9]:
pd.set_option("display.max_colwidth", None)
results_df = pd.DataFrame(results)

# Correct query for each question
correct_sql = {
    0: "SELECT count(*) FROM fact_loadsmart WHERE load_was_cancelled = FALSE AND delivery_date >= DATE '2024-12-01' AND delivery_date < DATE '2025-01-01'",
    1: "SELECT s.shipper_name FROM fact_loadsmart f JOIN dim_shippers s ON f.shipper_key = s.shipper_key GROUP BY 1 ORDER BY sum(f.book_price) DESC LIMIT 1",
    2: "SELECT l.pickup_state, avg(f.book_price) FROM fact_loadsmart f JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE f.load_was_cancelled = FALSE GROUP BY 1",
    3: "SELECT l.lane, count(*) FROM fact_loadsmart f JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE f.load_was_cancelled = FALSE GROUP BY 1 ORDER BY 2 DESC LIMIT 5",
    4: "SELECT c.carrier_name, count(*) FROM fact_loadsmart f JOIN dim_carriers c ON f.carrier_key = c.carrier_key JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE l.delivery_state = 'TX' GROUP BY 1 ORDER BY 2 DESC LIMIT 1",
    5: "SELECT CASE WHEN l.pickup_state = l.delivery_state THEN 'Intrastate' ELSE 'Interstate' END, avg(f.book_price) FROM fact_loadsmart f JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE f.load_was_cancelled = FALSE GROUP BY 1",
    6: """
        WITH top_shipper AS (
            SELECT shipper_key FROM fact_loadsmart WHERE load_was_cancelled = FALSE
            GROUP BY 1 ORDER BY count(*) DESC LIMIT 1
        )
        SELECT date_trunc('month', f.delivery_date), count(*)
        FROM fact_loadsmart f JOIN top_shipper t ON f.shipper_key = t.shipper_key
        WHERE f.load_was_cancelled = FALSE
        GROUP BY 1 ORDER BY 1
    """,
    7: "SELECT l.lane, avg(f.book_price) FROM fact_loadsmart f JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE f.load_was_cancelled = FALSE GROUP BY 1 HAVING count(*) >= 10 ORDER BY 2 DESC LIMIT 1",
}

def values(records):
    out = set()
    for row in records:
        for v in row.values():
            if isinstance(v, float):
                v = round(v, 2)
            out.add(str(v))
    return out

for i, sql in correct_sql.items():
    correct_answer = db_connection.execute(sql).fetchdf().to_dict("records")
    ai_answer = results_df.loc[i, "answer"]
    is_correct = values(correct_answer) <= values(ai_answer)
    results_df.loc[i, "correct?"] = "Yes" if is_correct else "No"
    results_df.loc[i, "notes"] = "no notes" if is_correct else "error"
    results_df.at[i, "answer"] = correct_answer

results_df["generated_sql"] = results_df["generated_sql"].str.split().str.join(" ")
results_df

,question,generated_sql,answer,correct?,notes
0,How many loads were delivered in the last full month available in the data?,SELECT COUNT(*) AS delivered_loads FROM fact_loadsmart WHERE delivery_date >= DATE '2024-12-01' AND delivery_date < DATE '2025-01-01' AND load_was_cancelled = FALSE;,[{'count_star()': 442}],Yes,no notes
1,Which shipper had the highest total book price?,SELECT ds.shipper_name FROM dim_shippers ds JOIN fact_loadsmart fl ON ds.shipper_key = fl.shipper_key WHERE fl.load_was_cancelled = FALSE GROUP BY ds.shipper_name ORDER BY SUM(fl.book_price) DESC LIMIT 1;,[{'shipper_name': 'Shipper 1249'}],Yes,no notes
2,What is the average book price per load by pickup state?,"SELECT l.pickup_state, AVG(f.book_price) AS average_book_price FROM fact_loadsmart f JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE f.load_was_cancelled = FALSE GROUP BY l.pickup_state;","[{'pickup_state': 'TX', 'avg(f.book_price)': 667.404943231443}, {'pickup_state': 'CA', 'avg(f.book_price)': 1980.4015155279506}, {'pickup_state': 'MI', 'avg(f.book_price)': 1684.6316}, {'pickup_state': 'NV', 'avg(f.book_price)': 1003.457045454546}, {'pickup_state': 'NC', 'avg(f.book_price)': 2150.8327102803732}, {'pickup_state': 'IN', 'avg(f.book_price)': 2081.307023809524}, {'pickup_state': 'NE', 'avg(f.book_price)': 1026.9947368421053}, {'pickup_state': 'IL', 'avg(f.book_price)': 1977.5470930232552}, {'pickup_state': 'WI', 'avg(f.book_price)': 2121.4463414634147}, {'pickup_state': 'GA', 'avg(f.book_price)': 1266.4133082706765}, {'pickup_state': 'PA', 'avg(f.book_price)': 1138.3525991189426}, {'pickup_state': 'NY', 'avg(f.book_price)': 1717.4918699186992}, {'pickup_state': 'CO', 'avg(f.book_price)': 1725.541967213115}, {'pickup_state': 'MA', 'avg(f.book_price)': 550.9035443037974}, {'pickup_state': 'MD', 'avg(f.book_price)': 917.6484615384616}, {'pickup_state': 'ID', 'avg(f.book_price)': 2076.401428571429}, {'pickup_state': 'WA', 'avg(f.book_price)': 1296.2576229508202}, {'pickup_state': 'WV', 'avg(f.book_price)': 924.4899999999999}, {'pickup_state': 'VA', 'avg(f.book_price)': 1480.4966666666667}, {'pickup_state': 'CT', 'avg(f.book_price)': 667.8711111111111}, {'pickup_state': 'MS', 'avg(f.book_price)': 2100.505}, {'pickup_state': 'AR', 'avg(f.book_price)': 2632.866666666667}, {'pickup_state': 'OH', 'avg(f.book_price)': 1916.4803157894737}, {'pickup_state': 'AL', 'avg(f.book_price)': 1106.255357142857}, {'pickup_state': 'OR', 'avg(f.book_price)': 2343.007358490566}, {'pickup_state': 'IA', 'avg(f.book_price)': 2528.2314285714283}, {'pickup_state': 'MN', 'avg(f.book_price)': 2338.185263157895}, {'pickup_state': 'NM', 'avg(f.book_price)': 2220.6133333333332}, {'pickup_state': 'UT', 'avg(f.book_price)': 1743.8066666666666}, {'pickup_state': 'KS', 'avg(f.book_price)': 1796.464}, {'pickup_state': 'WY', 'avg(f.book_price)': 1278.42}, {'pickup_state': 'NH', 'avg(f.book_price)': 1844.4709090909091}, {'pickup_state': 'TN', 'avg(f.book_price)': 1727.7017543859654}, {'pickup_state': 'NJ', 'avg(f.book_price)': 1192.580582524272}, {'pickup_state': 'KY', 'avg(f.book_price)': 1588.5076470588235}, {'pickup_state': 'SC', 'avg(f.book_price)': 2872.1869879518067}, {'pickup_state': 'RI', 'avg(f.book_price)': 626.6700000000001}, {'pickup_state': 'FL', 'avg(f.book_price)': 1525.376359649123}, {'pickup_state': 'LA', 'avg(f.book_price)': 1303.4020454545455}, {'pickup_state': 'MO', 'avg(f.book_price)': 2182.5561417322833}, {'pickup_state': 'AZ', 'avg(f.book_price)': 2212.8404545454546}, {'pickup_state': 'DE', 'avg(f.book_price)': 2242.485714285714}, {'pickup_state': 'OK', 'avg(f.book_price)': 1199.28}]",Yes,no notes
3,What are the top 5 lanes by number of delivered loads?,"SELECT l.lane, COUNT(*) AS delivered_loads FROM fact_loadsmart f JOIN dim_lanes l ON f.lane_key = l.lane_key WHERE f.load_was_cancelled = FALSE GROUP BY l.lane ORDER BY delivered_loads DESC LIMIT 5;","[{'lane': 'Hawkins,TX -> Roanoke,TX', 'count_star()': 882}, {'lane': 'Lodi,CA -> Pacific,WA', 'count_star()': 150}, {'l

## Iteration log

The first run used qwen2.5-coder:1.5b and failed 6 of 8 questions, always failing on SQL syntax, and making up columns that weren't even in the .yml. Switched to qwen2.5-coder:7b, which fixed most of the problems.

Three things still needed fixing after that:
- Question 1 still used current date instead of using the data's date range. Fixed it by improving the schema.yml (saying that the delivery_date is historical data).
- Question 7 returned one month instead of the full range. Ended up fixing itself once other columns were better documented, no specific rule was needed.
- Question 8 crashed because of a syntax error in the generated SQL. I was only able to fix it by making it retry once when a query fails (on function ask_ai_db_question).

Then I found a modelling problem. `pickup_city`, `pickup_state`, `delivery_city` and `delivery_state` were sitting in both `fact_loadsmart` and `dim_lanes`, which is not how a star schema should look. The fact should only carry `lane_key`. When I took them out of the fact so only `dim_lanes` had them, qwen2.5-coder:7b dropped from 8/8 to 6/8. It kept writing `pickup_state` straight off the fact table instead of joining `dim_lanes`. I tried three rounds of extra documentation and each round fixed one question and broke another, ending with queries that ran fine but gave quietly wrong answers.

So I checked if it was a documentation problem or a model problem. I removed those four columns from the schema context and asked qwen2.5-coder:7b, duckdb-nsql and llama3 the same question. All three still wrote `pickup_state` on the fact table, even though it was not in the context they were given. That is not a documentation gap, that is the model guessing which table a column belongs to.

Moved up to qwen2.5-coder:14b and it did the join with no trouble. Final version is the clean star schema, geography only in `dim_lanes`, running on 14b, 8/8 correct.

Assumption Made: Question 1 asked for "the last full month available in the data." I read that as December 2024, since the data barely has anything after that. I also wrote that assumption into the 'delivery_date' description in 'schema.yml'.

## Two questions a stakeholder would actually ask

- **What is the average profit per load by equipment type?** Profitability differs a lot by trailer type (flatbed vs reefer vs dry van), so this tells finance/ops where the margin actually comes from.
- **What is the total book price across all loads?** Basically total revenue tracking. Probably one of the first numbers anyone would ask for.

In [10]:
stakeholder_questions = [
    "What is the average profit per load by equipment type?",
    "What is the total book price across all loads?",
]

for question in stakeholder_questions:
    print(f"Q: {question}")
    sql_query, result_df = ask_ai_db_question(question)
    display(result_df)
    print()

Q: What is the average profit per load by equipment type?


--- SQL ---
SELECT 
    equipment_type, 
    AVG(pnl) AS average_profit_per_load
FROM 
    fact_loadsmart
WHERE 
    load_was_cancelled = FALSE
GROUP BY 
    equipment_type;
 ---


,equipment_type,average_profit_per_load
0,DRV,41.249418
1,FBE,224.766667
2,RFR,10.447327



Q: What is the total book price across all loads?


--- SQL ---
SELECT SUM(book_price) AS total_book_price
FROM fact_loadsmart
WHERE load_was_cancelled = FALSE;
 ---


,total_book_price
0,7106241.76


## A question the model can't answer

**What is the percentage of deliveries made on time by carrier?**

The schema has a 'carrier_on_time_to_delivery' column for this, and it is documented as NULL when "on-time" tracking wasn't recorded for a load. I wanted to see if the model and documentation were able to handle that NULL case correctly.

In [11]:
no_answer_question = "What is the percentage of deliveries made on time by carrier?"
sql_query, result_df = ask_ai_db_question(no_answer_question)
display(result_df)

--- SQL ---
SELECT 
    (SUM(CASE WHEN carrier_on_time_to_delivery = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS on_time_delivery_percentage
FROM 
    fact_loadsmart
WHERE 
    load_was_cancelled = FALSE;
 ---


,on_time_delivery_percentage
0,91.241479


It answered, and this time it did group by carrier, but the numbers are wrong. The query puts every delivered load in the denominator, including the ones where on-time tracking was never recorded. Those NULLs end up being counted as late.

Carrier 968589 is a good example. It has 17 delivered loads and 9 of them were never tracked, so the model reports 29.4% when the real number, counting only the loads that were actually tracked, is 62.5%. Any carrier with untracked loads looks worse than it really is, and nothing in the output shows that anything is missing.

### How this could be fixed

Improve the 'carrier_on_time_to_delivery' description in 'schema.yml' to say that NULL means tracking was never recorded, and that those loads have to be left out of the calculation completely instead of counted as late. AVG() already skips NULLs on its own, so this is the simple, expected query:

In [12]:
fixed_sql = """
    SELECT c.carrier_name, AVG(CAST(f.carrier_on_time_to_delivery AS INT)) AS on_time_delivery_percentage
    FROM fact_loadsmart f
    JOIN dim_carriers c ON f.carrier_key = c.carrier_key
    WHERE f.load_was_cancelled = FALSE
    GROUP BY c.carrier_name
"""
db_connection.execute(fixed_sql).fetchdf()

,carrier_name,on_time_delivery_percentage
0,Carrier 28008,1.0
1,Carrier 1401911,1.0
2,Carrier 1343372,1.0
3,Carrier 1695624,1.0
4,Carrier 1027120,1.0
...,...,...
2189,Carrier 967405,1.0
2190,Carrier 1215497,1.0
2191,Carrier 80899,1.0
2192,Carrier 127275,1.0
